In [0]:
%sql

-- Create table dim_event
create or replace table marathos.gold.dim_event as
select distinct
    event_id,
    event_name,
    event_dates,
    event_distance_length,
    event_number_of_finishers,
    case 
        when event_distance_length rlike '(?i)h$'
            then 'Time'
        when event_distance_length rlike '(?i)(km|mi|miles)'
            then 'Distance'
        else 'Unknown'
    end as event_type
from
    marathos.silver.marathon_obt

In [0]:
%sql

-- Create table dim_athlete
create table marathos.gold.dim_athlete as
select distinct
    athlete_id,
    athlete_country,
    athlete_year_of_birth,
    athlete_gender,
    athlete_age_category
from
    marathos.silver.marathon_obt

In [0]:
%sql

-- Create table dim_club
create table marathos.gold.dim_club as
select
    row_number() over (order by athlete_club) as club_id, -- Create club ID
    athlete_club
from (
    select distinct athlete_club
    from marathos.silver.marathon_obt
    where athlete_club is not null
)

In [0]:
%sql

-- Create table fct_results
create table marathos.gold.fct_results as
select
    row_number() over (order by s.event_id, s.athlete_id) as result_id,
    s.event_id,
    s.athlete_id,
    c.club_id,
    s.performance_seconds,
    s.athlete_average_speed

from marathos.silver.marathon_obt as s
left join marathos.gold.dim_club as c on s.athlete_club = c.athlete_club